In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

# Sample car data (use your own CSV for full runs)
data = pd.DataFrame({
    'Car_Name': ["Honda City", "Hyundai i20", "Ford Figo", "Toyota Innova", "Maruti Swift"],
    'Year': [2015, 2017, 2016, 2012, 2018],
    'Present_Price': [9.9, 8.5, 7.3, 12.5, 6.7],
    'Driven_kms': [40000, 35000, 37000, 65000, 15000],
    'Fuel_Type': ["Petrol", "Diesel", "Petrol", "Diesel", "Petrol"],
    'Seller_Type': ["Dealer", "Individual", "Dealer", "Dealer", "Individual"],
    'Transmission': ["Manual", "Manual", "Manual", "Manual", "Manual"],
    'Owner': [0, 0, 0, 1, 0],
    'Selling_Price': [6.5, 5.0, 4.75, 8.2, 5.8]
})

# For full runs with actual data, upload the CSV in Colab/Notebook and use:
# data = pd.read_csv('/your/path/yourfilename.csv')

data['CarAge'] = 2025 - data['Year']
data.drop(['Year', 'Car_Name'], axis=1, inplace=True)
data = data[data['Selling_Price'] > 0]

cat_features = ['Fuel_Type', 'Seller_Type', 'Transmission', 'Owner']
num_features = ['Present_Price', 'Driven_kms', 'CarAge']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

X = data.drop(['Selling_Price'], axis=1)
y = data['Selling_Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "LinearRegression": LinearRegression(),
    "RidgeRegression": Ridge(alpha=1.0),
    "LassoRegression": Lasso(alpha=0.1),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results[name] = {
        'R2': r2_score(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred))
    }

print("Model Comparison Table:")
for name, metrics in results.items():
    print(f"{name}: R2={metrics['R2']:.3f}, MAE={metrics['MAE']:.3f}, RMSE={metrics['RMSE']:.3f}")

# --- Example Visual Plot (Random Forest Feature Importances, requires enough data) ---
# If your dataset is small, this might be skipped — include once running with your full dataset.

# --- Demo Prediction for New User Input ---
def predict_price(user_input):
    input_df = pd.DataFrame([user_input])
    model = Pipeline([('preprocessor', preprocessor), ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])
    model.fit(X_train, y_train)
    return model.predict(input_df)[0]

sample_input = {
    'Present_Price': 9.5,
    'Driven_kms': 38000,
    'Fuel_Type': 'Petrol',
    'Seller_Type': 'Individual',
    'Transmission': 'Manual',
    'Owner': 0,
    'CarAge': 7
}

print("Predicted Price for sample input:", predict_price(sample_input))


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


Model Comparison Table:
LinearRegression: R2=nan, MAE=1.731, RMSE=1.731
RidgeRegression: R2=nan, MAE=1.343, RMSE=1.343
LassoRegression: R2=nan, MAE=1.301, RMSE=1.301
RandomForest: R2=nan, MAE=0.894, RMSE=0.894
Predicted Price for sample input: 5.701500000000001
